%md
## Gold Layer: Fact Table + Analytics Tables
Joins cleaned orders to SCD2 dimensions using a point-in-time date-range join (order_date must fall between effective_start_date and effective_end_date), then builds 4 aggregate tables for reporting.

In [0]:
%run "./00_setup"

%md
### Step 0: Setup
Creating catalog, schemas (raw/silver/gold), and a volume to store raw files.

Catalog and schemas created successfully


%md
### Load Silver Tables
Pulling in cleaned orders and all 3 dimensions before building the fact table.

In [0]:
from pyspark.sql import functions as F

orders = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver1_orders_clean")
dim_customer = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2")
dim_product = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2")
dim_store = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_store")

print("orders:", orders.count())
print("dim_customer:", dim_customer.count())
print("dim_product:", dim_product.count())
print("dim_store:", dim_store.count())

orders: 17273
dim_customer: 3093
dim_product: 960
dim_store: 75


%md
### Build Fact Table
Point-in-time join: order_date must fall between the dimension row's effective_start_date and effective_end_date. This resolves the correct historical version of the customer/product as of when the order happened — a plain ID match would incorrectly always give the current attributes.

In [0]:
orders_d = orders.withColumn("order_date", F.to_date("order_ts"))

fact_orders = (
    orders_d.alias("o")
    .join(
        dim_customer.alias("c"),
        (F.col("o.customer_id") == F.col("c.customer_id"))
        & (F.col("o.order_date") >= F.col("c.effective_start_date"))
        & (F.col("o.order_date") <= F.col("c.effective_end_date")),
        "left",
    )
    .join(
        dim_product.alias("p"),
        (F.col("o.product_id") == F.col("p.product_id"))
        & (F.col("o.order_date") >= F.col("p.effective_start_date"))
        & (F.col("o.order_date") <= F.col("p.effective_end_date")),
        "left",
    )
    .join(dim_store.alias("s"), F.col("o.store_id") == F.col("s.store_id"), "left")
    .select(
        "o.order_id", "o.order_date", "o.order_ts",
        "c.customer_sk", "o.customer_id",
        "p.product_sk", "o.product_id",
        "s.store_id",
        "o.quantity", "o.unit_price", "o.discount_pct", "o.gross_amount",
        "o.payment_method", "o.order_status", "o.coupon_code",
        "c.segment", "c.city",
        "p.category", "p.brand",
        "s.region",
    )
)

fact_orders.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")

print("fact_orders rows:", fact_orders.count())

fact_orders rows: 17273


##Sample rows from the conformed fact table

In [0]:
%sql
SELECT * FROM retail_demo.gold.fact_orders LIMIT 10

order_id,order_date,order_ts,customer_sk,customer_id,product_sk,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,coupon_code,segment,city,category,brand,region
O0000001,2025-12-15,2025-12-15T04:49:00.000Z,46695eac27bcfc1cd6ef628c1cd9e906ba842f332bcad364d40b1a2930606679,C01671,87eb8dc90d0c0ff0ed136a6970088973eb1fd9ed2f5a02871fee11412d282e87,P00638,S003,4,30260.29,0.15,121041.16,CARD,cancelled,null,Gold,Pune,Home,BrandD,South
O0000004,2025-12-28,2025-12-28T13:56:00.000Z,cd9084d058117a20ef23850a55eb353614492d94f319a47b886ab56e416e83d0,C01399,b1821d99d372f7687394ef518cdf0870fc85644d9ef245dfa294112b72340a67,P00031,S071,1,18705.7,0.05,15899.84,NETBANKING,returned,null,Regular,Jaipur,Beauty,BrandA,Online
O0000005,2025-12-31,2025-12-31T17:43:00.000Z,8b0314b89d7d52d1b91cd7805372bdf7fca4a59f9237609770db67a8c54604d4,C00383,166185d29b78aa61d60eb0fa564838d9df37eb7938888dfaca1cfa8633c87ba8,P00564,S049,2,46727.14,0.15,84108.85,COD,cancelled,null,Gold,Delhi,Grocery,BrandB,West
O0000007,2025-12-26,2025-12-26T03:16:00.000Z,f29cce2a246590ee8f95c0c30a10fc4295d492e82aff010088ab762e12c0acf3,C01633,9c5ac2d00f8f97a872591ea98cc62cf41abe57abe902ebaafc61bb9719f61a1a,P00555,S020,2,54612.57,0.05,98302.63,COD,returned,null,Gold,Gurugram,Electronics,BrandC,Online
O0000009,2026-01-20,2026-01-20T19:25:00.000Z,1c5cecc5c68c51e72255ca03b6463ab2a81488d988e8ef32380e495d4cefa4cf,C00479,62f086ff842cb3173a4c696e91f720e4c7e50e5e98294503899e6b8543bf3e6d,P00782,S050,4,84092.97,0.05,336371.88,UPI,delivered,null,Silver,Jaipur,Beauty,BrandC,East
O0000010,2026-03-21,2026-03-21T06:52:00.000Z,34d1d7517c9e09bc074bea836393325aad961cf764cecb37af417ac335495cd7,C00197,c8233df98d552c26f93545826b160ce5438f0a54d0a948835d0d968f5c1bb3d0,P00068,S049,5,31466.94,0.0,157334.7,COD,delivered,null,Platinum,Unknown,Fashion,BrandB,West
O0000011,2025-12-15,2025-12-15T20:38:00.000Z,086fb5969aafc8d4c1c130b4947aabb5e0c0e581d3247b1e5be062a5dec8dbfd,C00223,559d67241963ee7a2a993751ac054d137f666c2308ac577335c9f9be85f25450,P00465,S053,1,66406.1,0.05,59765.49,CARD,returned,null,Silver,Bengaluru,Home,BrandD,South
O0000012,2026-01-22,2026-01-22T20:26:00.000Z,8671cdcc736686cf24cbcca6b79dda37a66f89ad123e9b885aaa70a67ded80a8,C00904,72b94345f46df3faa98e28093b3b0abd85c243db3d76aa71ded89c526a0acd76,P00653,S036,2,41566.57,0.1,74819.83,UPI,cancelled,null,Regular,Hyderabad,Home,BrandD,North
O0000015,2025-11-11,2025-11-11T18:44:00.000Z,14d6c577d33d47cb8563854e87475785d1845a77ecb59e801361fe101b636d4d,C00094,085e35b24b2c6c24ab90cd7b584e23a9c56870505574085936dfde57d72ad8b8,P00407,S068,5,42833.72,0.1,192751.74,UPI,delivered,null,Platinum,Chennai,Electronics,BrandC,South
O0000016,2025-11-20,2025-11-20T06:29:00.000Z,73fba5efbd9341d4e4c1faa154040f3e32bbc8550c04a467253904be7c54c674,C01287,a653c7f9643519434640c06c513fc1e65effe0392604c8177ea6319209314e4c,P00367,S022,2,65681.17,0.05,124794.22,COD,cancelled,null,Regular,Ahmedabad,Electronics,BrandB,East


%md
### Gold Analytics Tables
Four aggregate tables built on top of fact_orders: daily sales, category sales, segment sales, region sales.

In [0]:
fact = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")

gold_daily_sales = (
    fact.groupBy("order_date")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("gross_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_units"),
        F.round(F.avg("gross_amount"), 2).alias("avg_order_value"),
    )
    .orderBy("order_date")
)
gold_daily_sales.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_daily_sales")

gold_category_sales = (
    fact.groupBy("category")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("gross_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_units"),
    )
    .orderBy(F.col("total_revenue").desc())
)
gold_category_sales.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_category_sales")

gold_segment_sales = (
    fact.groupBy("segment")
    .agg(
        F.countDistinct("customer_id").alias("unique_customers"),
        F.count("order_id").alias("total_orders"),
        F.sum("gross_amount").alias("total_revenue"),
    )
    .orderBy(F.col("total_revenue").desc())
)
gold_segment_sales.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_segment_sales")

gold_region_sales = (
    fact.groupBy("region")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("gross_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_units"),
    )
    .orderBy(F.col("total_revenue").desc())
)
gold_region_sales.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_region_sales")

print("all 4 gold tables created")
display(gold_daily_sales)
display(gold_category_sales)
display(gold_segment_sales)
display(gold_region_sales)

all 4 gold tables created


order_date,total_orders,total_revenue,total_units,avg_order_value
2025-10-01,60,7931198.779999998,193,132186.65
2025-10-02,49,5833780.93,135,119056.75
2025-10-03,57,6742787.449999999,160,118294.52
2025-10-04,64,8789390.049999999,201,137334.22
2025-10-05,53,6658672.449999999,159,125635.33
2025-10-06,61,8034282.170000002,188,131709.54
2025-10-07,63,8086661.39,174,128359.7
2025-10-08,63,6900786.519999998,185,109536.29
2025-10-09,49,6025594.72,136,122971.32
2025-10-10,65,7819608.3199999975,208,120301.67


category,total_orders,total_revenue,total_units
Home,3822,4.823690200500017E8,12096
Fashion,3599,4.632696704999988E8,11275
Grocery,2999,3.966898759199998E8,9355
Electronics,3120,3.952955608099997E8,9790
Beauty,2901,3.661724408300001E8,9089
null,548,6.795196089999993E7,1633
Unknown,284,3.716335242E7,865


segment,unique_customers,total_orders,total_revenue
Platinum,689,4298,5.478114503299999E8
Regular,662,4222,5.400572678199997E8
Silver,648,4110,5.2701413831999964E8
Gold,657,3923,5.0589084235000074E8
null,283,720,8.813818261000003E7


region,total_orders,total_revenue,total_units
Online,4408,5.609071000899991E8,13667
South,3950,5.070504159600001E8,12365
North,3295,4.289448267199995E8,10451
West,3177,4.078078436800004E8,10074
East,2443,3.042016949800008E8,7546


%md
### Delta Lake Features
Demonstrating table history (transaction log) and time travel - core Delta Lake capabilities used throughout this pipeline.

In [0]:
%sql
DESCRIBE HISTORY retail_demo.gold.fact_orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-07-13T10:48:16.000Z,78202286519270,nanus560.ns@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1263003088367415),7b7f6c4c-1a2d-4e57-a1ae-30d4b3ff8096,0713-103307-9q8jbkd9-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 555493, numDeletionVectorsRemoved -> 0, numOutputRows -> 17273, numOutputBytes -> 555493)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-12T09:13:57.000Z,78202286519270,nanus560.ns@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1263003088367415),3abb5ae9-7093-4f26-9176-2abe548534e3,0712-081419-g299aodl-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 17273, numOutputBytes -> 555493)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")
print("Version 0 row count:", df_v0.count())

df_latest = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")
print("Latest version row count:", df_latest.count())

Version 0 row count: 17273
Latest version row count: 17273


%md
### Schema Evolution (Already Demonstrated)
The `coupon_code` column appeared only in day-3 incremental orders. Auto Loader's `cloudFiles.schemaEvolutionMode = "addNewColumns"` and Delta's `mergeSchema = "true"` option (used in 01_bronze) allowed this new column to be accepted automatically without failing the pipeline — visible in `bronze_orders_incremental`, where day-1/day-2 rows have `coupon_code = null` and day-3 rows have actual values.

%md
### Sanity Check — Gold Layer
Verify fact table join quality (no unexpected nulls from SCD2 join) and preview all 4 analytics tables.

In [0]:
print("orders with missing customer_sk:", fact.filter("customer_sk is null").count())
print("orders with missing product_sk:", fact.filter("product_sk is null").count())
print("orders with missing store:", fact.filter("store_id is null").count())

print("\ntotal gross_amount in fact_orders:", fact.agg(F.sum("gross_amount")).collect()[0][0])

print("\n--- gold_daily_sales ---")
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.gold_daily_sales").limit(5))

print("--- gold_category_sales ---")
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.gold_category_sales"))

print("--- gold_segment_sales ---")
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.gold_segment_sales"))

print("--- gold_region_sales ---")
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.gold_region_sales"))

orders with missing customer_sk: 720
orders with missing product_sk: 536
orders with missing store: 0

total gross_amount in fact_orders: 2208911881.4300117

--- gold_daily_sales ---


order_date,total_orders,total_revenue,total_units,avg_order_value
2025-10-01,60,7931198.779999998,193,132186.65
2025-10-02,49,5833780.93,135,119056.75
2025-10-03,57,6742787.449999999,160,118294.52
2025-10-04,64,8789390.049999999,201,137334.22
2025-10-05,53,6658672.449999999,159,125635.33


--- gold_category_sales ---


category,total_orders,total_revenue,total_units
Home,3822,4.823690200500017E8,12096
Fashion,3599,4.632696704999988E8,11275
Grocery,2999,3.966898759199998E8,9355
Electronics,3120,3.952955608099997E8,9790
Beauty,2901,3.661724408300001E8,9089
null,548,6.795196089999993E7,1633
Unknown,284,3.716335242E7,865


--- gold_segment_sales ---


segment,unique_customers,total_orders,total_revenue
Platinum,689,4298,5.478114503299999E8
Regular,662,4222,5.400572678199998E8
Silver,648,4110,5.270141383200002E8
Gold,657,3923,5.058908423500001E8
null,283,720,8.813818260999998E7


--- gold_region_sales ---


region,total_orders,total_revenue,total_units
Online,4408,5.609071000899991E8,13667
South,3950,5.070504159600001E8,12365
North,3295,4.289448267199995E8,10451
West,3177,4.078078436800004E8,10074
East,2443,3.042016949800008E8,7546
